In [2]:
import pandas as pd
import numpy as np
from darts import TimeSeries
from darts.models import RegressionModel
from darts.utils.timeseries_generation import datetime_attribute_timeseries
from darts.dataprocessing.transformers import Scaler
from darts.metrics import mape
from sklearn.model_selection import TimeSeriesSplit
from joblib import Parallel, delayed
from xgboost import XGBRegressor

#Load the engineered data
df = pd.read_parquet('../../Data/Merged_Data/eaglei_noaa_era5_engineered.parquet')

# Restrict df to years prior to 2022 and after 2019
df = df[df['YEAR'] < 2022]
df = df[df['YEAR'] > 2019]

# We'll focus on Maine, Oregon, Arizona, Nebraska, and South Carolina
df = df[df['STUSPS'].isin(['ME', 'OR', 'AZ', 'NE', 'SC'])]

# Create a new variable 'customers_out_prop" by dividing customers_out by POPULATION
df['customers_out_prop'] = df['customers_out'] / df['POPULATION']

# Create a new variable dividing POPULATION divided by AREA
df['density'] = df['POPULATION'] / df['AREA']

# Note: Not currently including Subregion b/c we'd need to convert to a dummy variable

features_ts = ['event_count SnowIce',
       'event_count Flood', 'event_count Storm', 'event_count Hurricane',
       'event_count Heat', 'event_count Fire', 'event_count Wind',
       'event_count Ocean', 'event_count Other', 't2m', 'sf', 'tp', 't2m_mean',
       't2m_max', 'sf_mean', 'sf_max', 'tp_mean', 'tp_max', 'wind_speed',
       'neighbor_mean_wind_speed', 'neighbor_max_wind_speed', 'sf_12h',
       'sf_24h', 'tp_12h', 'tp_24h']

features_static = ['Pct_Buried_Lines','centroid_longitude','centroid_latitude','density', 'POPULATION']

# Your variable lists
target_col = 'customers_out_prop'
time_col = 'datetime'
id_col = 'fips_code'
forecast_horizon = 4        # 4 timesteps (6-hourly cadence = 24h)

# --- Prepare data grouped by county ---
grouped = dict(tuple(df.groupby(id_col)))

# Convert to Darts TimeSeries objects
series_list = []
past_cov_list = []
static_cov_list = []

for fips, group in grouped.items():
    group = group.sort_values(time_col)
    ts_target = TimeSeries.from_dataframe(group, time_col, target_col)
    ts_covariates = TimeSeries.from_dataframe(group, time_col, features_ts)
    
    series_list.append(ts_target)
    past_cov_list.append(ts_covariates)
    
    # Static covariates are same for all time points in group
    static_cov = group[features_constant].iloc[0]
    static_cov_list.append(static_cov)

# --- Scaling ---
target_scaler = Scaler()
cov_scaler = Scaler()

series_scaled = target_scaler.fit_transform(series_list)
cov_scaled = cov_scaler.fit_transform(past_cov_list)

# --- Assign static covariates ---
for ts, static_cov in zip(series_scaled, static_cov_list):
    ts.static_covariates = static_cov

# --- Transform target series into max-over-next-4-horizon task ---
def to_rolling_max_target(series, horizon):
    y = []
    for i in range(len(series) - horizon):
        y.append(np.max(series[i+1:i+1+horizon].values()))
    time_index = series[:-horizon].time_index
    return TimeSeries.from_times_and_values(time_index, y, static_covariates=series.static_covariates)

target_max_series = [to_rolling_max_target(ts, forecast_horizon) for ts in series_scaled]
cov_trimmed = [ts[:-forecast_horizon] for ts in cov_scaled]

# --- Model setup ---
model = RegressionModel(
    lags=20,
    lags_past_covariates=20,
    model=XGBRegressor(),
    use_static_covariates=True,
    output_chunk_length=1,
    multi_models=False,
    random_state=42,
)

# --- Cross-validation setup ---
tscv = TimeSeriesSplit(n_splits=5)

def run_cv(train_series, train_covs, val_series, val_covs):
    model.fit(
        series=train_series,
        past_covariates=train_covs,
        verbose=False,
    )
    forecasts = model.predict(n=1, series=val_series, past_covariates=val_covs)
    return forecasts

# --- Parallelized cross-validation ---
results = []

for train_idx, val_idx in tscv.split(target_max_series[0]):
    train_series = [ts[train_idx] for ts in target_max_series]
    val_series = [ts[val_idx] for ts in target_max_series]
    train_covs = [ts[train_idx] for ts in cov_trimmed]
    val_covs = [ts[val_idx] for ts in cov_trimmed]
    
    # Each fold runs in parallel
    fold_results = Parallel(n_jobs=-1)(
        delayed(run_cv)(
            [train_series[i] for i in range(len(series_scaled))],
            [train_covs[i] for i in range(len(series_scaled))],
            [val_series[i] for i in range(len(series_scaled))],
            [val_covs[i] for i in range(len(series_scaled))],
        )
    )
    results.append(fold_results)

# --- Evaluate ---
def evaluate(preds, actuals):
    return np.mean([mape(p, a) for p, a in zip(preds, actuals)])

# Example: evaluate last fold
preds = results[-1]
actuals = [ts[val_idx][-1:] for ts in target_max_series]
print("Last fold MAPE:", evaluate(preds, actuals))


KeyboardInterrupt: 

In [29]:
import pandas as pd
import numpy as np
from darts import TimeSeries
from darts.models import RegressionModel
from darts.dataprocessing.transformers import Scaler
from darts.metrics import mape
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor
from joblib import Parallel, delayed
from tqdm import tqdm


#Load the engineered data
df = pd.read_parquet('../../Data/Merged_Data/eaglei_noaa_era5_engineered.parquet')

# Restrict df to years prior to 2022 and after 2019
df = df[df['YEAR'] < 2022]
df = df[df['YEAR'] > 2019]

# We'll focus on Maine, Oregon, Arizona, Nebraska, and South Carolina
df = df[df['STUSPS'].isin(['ME', 'OR', 'AZ', 'NE', 'SC'])]

# Create a new variable 'customers_out_prop" by dividing customers_out by POPULATION
df['customers_out_prop'] = df['customers_out'] / df['POPULATION']

# Create a new variable dividing POPULATION divided by AREA
df['density'] = df['POPULATION'] / df['AREA']

# Note: Not currently including Subregion b/c we'd need to convert to a dummy variable

features_ts = ['event_count SnowIce',
       'event_count Flood', 'event_count Storm', 'event_count Hurricane',
       'event_count Heat', 'event_count Fire', 'event_count Wind',
       'event_count Ocean', 'event_count Other', 't2m', 'sf', 'tp', 't2m_mean',
       't2m_max', 'sf_mean', 'sf_max', 'tp_mean', 'tp_max', 'wind_speed',
       'neighbor_mean_wind_speed', 'neighbor_max_wind_speed', 'sf_12h',
       'sf_24h', 'tp_12h', 'tp_24h']

features_static = ['Pct_Buried_Lines','centroid_longitude','centroid_latitude','density', 'POPULATION']

# --- CONFIG ---
forecast_horizon = 4
lags = 20
n_splits = 5

# --- PREP DATA ---
grouped = dict(tuple(df.groupby('fips_code')))
fips_codes = list(grouped.keys())

series_list, cov_list, static_list = [], [], []

for fips, grp in grouped.items():
    grp = grp.sort_values('datetime')
    ts_target = TimeSeries.from_dataframe(grp, 'datetime', 'customers_out_prop')
    ts_covs = TimeSeries.from_dataframe(grp, 'datetime', features_ts)
    
    series_list.append(ts_target)
    cov_list.append(ts_covs)
    static_list.append(grp[features_static].iloc[0])

# Scale
target_scaler = Scaler()
cov_scaler = Scaler()
series_scaled = target_scaler.fit_transform(series_list)
cov_scaled = cov_scaler.fit_transform(cov_list)

# Attach static covariates
for ts, static in zip(series_scaled, static_list):
    ts = ts.with_static_covariates(static)

# Create max-over-horizon targets
def to_max_series(series, horizon):
    y = [np.max(series[i+1:i+1+horizon].values()) for i in range(len(series)-horizon)]
    time_index = series[:-horizon].time_index
    return TimeSeries.from_times_and_values(time_index, y, static_covariates=series.static_covariates)

target_max = [to_max_series(ts, forecast_horizon) for ts in series_scaled]
cov_trimmed = [ts[:-forecast_horizon] for ts in cov_scaled]

# --- TIME SPLITS ---
tscv = TimeSeriesSplit(n_splits=n_splits)
indices = list(tscv.split(target_max[0]))

# --- RESULTS CONTAINER ---
results = []
final_model = None

# --- CROSS-VALIDATION LOOP ---
for fold, (train_idx, val_idx) in enumerate(tqdm(indices, desc="CV Folds")):
    # Use a reference series to get actual time ranges
    reference_times = target_max[0].time_index
    train_start = reference_times[train_idx[0]]
    train_end = reference_times[train_idx[-1]]
    val_start = reference_times[val_idx[0]]
    val_end = reference_times[val_idx[-1]]

    # Slice all series by time
    train_series = [ts.slice(train_start, train_end) for ts in target_max]
    val_series = [ts.slice(val_start, val_end) for ts in target_max]
    train_covs = [ts.slice(train_start, train_end) for ts in cov_trimmed]
    val_covs = [ts.slice(val_start, val_end) for ts in cov_trimmed]

    model = RegressionModel(
        model=XGBRegressor(n_jobs=-1, random_state=random_state),
        lags=lags,
        lags_past_covariates=lags,
        output_chunk_length=forecast_horizon,
        use_static_covariates=True,
        multi_models=False,
    )

    # Fit once (global)
    model.fit(series=train_series, past_covariates=train_covs, verbose=False)
    final_model = model  # Store last model

    # --- PREDICT PARALLELIZED ---
    def predict_one(i):
        fips = fips_codes[i]
        full_series = series_list[i]  # full target series (train + val)
        covariates = features_ts[i]  # time-varying covariates
        static = features_static[i]    # static covariates
        
        full_series = full_series.with_static_covariates(static)

        results = []

        # Use val_series to define valid forecast windows
        val_start = val_series[i].start_time()
        val_end = val_series[i].end_time()

        timestamps = pd.date_range(
            val_start, 
            val_end - forecast_horizon * full_series.freq, 
            freq=full_series.freq
        )

        print(f"\n[DEBUG] fips={fips}, rolling predictions: {len(timestamps)}")

        for t in timestamps:
            try:
                # Input = all data before t
                input_ts = full_series.drop_after(t)
                input_covs = covariates.drop_after(t)

                # Forecast n steps from that point
                pred = model.predict(
                    n=forecast_horizon,
                    series=input_ts,
                    past_covariates=input_covs,
                    show_warnings=False
                )

                if pred is None or len(pred) < forecast_horizon:
                    continue

                pred_max = float(pred.values().max())

                # True values for same time window
                true_window = full_series.slice(t + full_series.freq, t + forecast_horizon * full_series.freq)

                if len(true_window) < forecast_horizon:
                    continue

                true_max = float(true_window.values().max())

                results.append({
                    "fips_code": fips,
                    "fold": fold,
                    "predicted_max": pred_max,
                    "true_max": true_max,
                    "datetime": t.to_pydatetime()
                })

            except Exception as e:
                print(f"[ERROR] fips={fips}, t={t}: {e}")
                continue

        print(f"[DONE] fips={fips}, predictions: {len(results)}")
        return results





    fold_results = Parallel(n_jobs=-1, backend="loky")(delayed(predict_one)(i) for i in range(len(series_list)))
    results.extend([item for sublist in fold_results for item in sublist])

# --- BUILD RESULTS DATAFRAME ---
cv_results_df = pd.DataFrame(results)

# --- FINAL REGRESSOR PARAMS ---
regressor_params = final_model.model.get_params()

# Output
cv_results_df.head(), regressor_params


CV Folds:   0%|          | 0/5 [00:00<?, ?it/s]ValueError: `static_covariates` must be either a pandas Series, DataFrame or None
ValueError: `static_covariates` must be either a pandas Series, DataFrame or None
ValueError: `static_covariates` must be either a pandas Series, DataFrame or None
ValueError: `static_covariates` must be either a pandas Series, DataFrame or None
ValueError: `static_covariates` must be either a pandas Series, DataFrame or None
CV Folds:   0%|          | 0/5 [00:11<?, ?it/s]


IndexError: list index out of range

In [27]:
cv_results_df

""


In [ ]:
import pandas as pd
import numpy as np
from darts import TimeSeries
from darts.models import RegressionModel
from darts.dataprocessing.transformers import Scaler
from darts.metrics import mape
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor
from joblib import Parallel, delayed
from tqdm import tqdm


#Load the engineered data
df = pd.read_parquet('../../Data/Merged_Data/eaglei_noaa_era5_engineered.parquet')

# Restrict df to years prior to 2022 and after 2019
df = df[df['YEAR'] < 2022]
df = df[df['YEAR'] > 2019]

# We'll focus on Maine, Oregon, Arizona, Nebraska, and South Carolina
df = df[df['STUSPS'].isin(['ME', 'OR', 'AZ', 'NE', 'SC'])]

# Create a new variable 'customers_out_prop" by dividing customers_out by POPULATION
df['customers_out_prop'] = df['customers_out'] / df['POPULATION']

# Create a new variable dividing POPULATION divided by AREA
df['density'] = df['POPULATION'] / df['AREA']

# Note: Not currently including Subregion b/c we'd need to convert to a dummy variable

features_ts = ['event_count SnowIce',
       'event_count Flood', 'event_count Storm', 'event_count Hurricane',
       'event_count Heat', 'event_count Fire', 'event_count Wind',
       'event_count Ocean', 'event_count Other', 't2m', 'sf', 'tp', 't2m_mean',
       't2m_max', 'sf_mean', 'sf_max', 'tp_mean', 'tp_max', 'wind_speed',
       'neighbor_mean_wind_speed', 'neighbor_max_wind_speed', 'sf_12h',
       'sf_24h', 'tp_12h', 'tp_24h']

features_static = ['Pct_Buried_Lines','centroid_longitude','centroid_latitude','density', 'POPULATION']

# --- CONFIG ---
forecast_horizon = 4
lags = 20
n_splits = 5

# --- PREP DATA ---
grouped = dict(tuple(df.groupby('fips_code')))
fips_codes = list(grouped.keys())

series_list, cov_list, static_list = [], [], []

for fips, grp in grouped.items():
    grp = grp.sort_values('datetime')
    ts_target = TimeSeries.from_dataframe(grp, 'datetime', 'customers_out_prop')
    ts_covs = TimeSeries.from_dataframe(grp, 'datetime', features_ts)
    
    series_list.append(ts_target)
    cov_list.append(ts_covs)
    static_list.append(grp[features_static].iloc[0])

# Scale
target_scaler = Scaler()
cov_scaler = Scaler()
series_scaled = target_scaler.fit_transform(series_list)
cov_scaled = cov_scaler.fit_transform(cov_list)

# Attach static covariates
for ts, static in zip(series_scaled, static_list):
    ts = ts.with_static_covariates(static)

# Create max-over-horizon targets
def to_max_series(series, horizon):
    y = [np.max(series[i+1:i+1+horizon].values()) for i in range(len(series)-horizon)]
    time_index = series[:-horizon].time_index
    return TimeSeries.from_times_and_values(time_index, y, static_covariates=series.static_covariates)

target_max = [to_max_series(ts, forecast_horizon) for ts in series_scaled]
cov_trimmed = [ts[:-forecast_horizon] for ts in cov_scaled]

# --- TIME SPLITS ---
tscv = TimeSeriesSplit(n_splits=n_splits)
indices = list(tscv.split(target_max[0]))

# --- RESULTS CONTAINER ---
results = []
final_model = None

# --- CROSS-VALIDATION LOOP ---
for fold, (train_idx, val_idx) in enumerate(tqdm(indices, desc="CV Folds")):
    # Use a reference series to get actual time ranges
    reference_times = target_max[0].time_index
    train_start = reference_times[train_idx[0]]
    train_end = reference_times[train_idx[-1]]
    val_start = reference_times[val_idx[0]]
    val_end = reference_times[val_idx[-1]]

    # Slice all series by time
    train_series = [ts.slice(train_start, train_end) for ts in target_max]
    val_series = [ts.slice(val_start, val_end) for ts in target_max]
    train_covs = [ts.slice(train_start, train_end) for ts in cov_trimmed]
    val_covs = [ts.slice(val_start, val_end) for ts in cov_trimmed]

    model = RegressionModel(
        model=XGBRegressor(n_jobs=-1, random_state=random_state),
        lags=lags,
        lags_past_covariates=lags,
        output_chunk_length=forecast_horizon,
        use_static_covariates=True,
        multi_models=False,
    )

    # Fit once (global)
    model.fit(series=train_series, past_covariates=train_covs, verbose=False)
    final_model = model  # Store last model

    # --- PREDICT PARALLELIZED ---
    def predict_one(i):
        fips = fips_codes[i]
        ts_input = train_series[i]
        cov_full = cov_trimmed[i]

        try:
            print(f"\n[DEBUG] Testing prediction for fips_code={fips}")

            # Basic test prediction — no start time
            pred = model.predict(
                n=forecast_horizon,
                series=ts_input,
                past_covariates=cov_full,
                show_warnings=False
            )

            print(f"[DEBUG] Got prediction for fips_code={fips}: length={len(pred)}")
            print(f"[DEBUG] Prediction values:\n{pred.values()}")

            return [{
                "fips_code": fips,
                "fold": fold,
                "predicted_max": float(pred.values().max()),
                "true_max": None,
                "datetime": ts_input.end_time().to_pydatetime()
            }]

        except Exception as e:
            print(f"[ERROR] fips_code={fips}: {e}")
            return []


    fold_results = Parallel(n_jobs=-1, backend="loky")(delayed(predict_one)(i) for i in range(len(series_list)))
    results.extend([item for sublist in fold_results for item in sublist])

# --- BUILD RESULTS DATAFRAME ---
cv_results_df = pd.DataFrame(results)

# --- FINAL REGRESSOR PARAMS ---
regressor_params = final_model.model.get_params()

# Output
cv_results_df.head(), regressor_params


Exception ignored in: <function ResourceTracker.__del__ at 0x106fa6a20>
Traceback (most recent call last):
  File "/opt/homebrew/anaconda3/envs/erdos_spring_2025/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/homebrew/anaconda3/envs/erdos_spring_2025/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/opt/homebrew/anaconda3/envs/erdos_spring_2025/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x106a5ea20>
Traceback (most recent call last):
  File "/opt/homebrew/anaconda3/envs/erdos_spring_2025/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/homebrew/anaconda3/envs/erdos_spring_2025/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/opt/homebrew/anaconda3/envs/erdos_spring_2025/lib/python3.12/multiprocessing/resource_


[DEBUG] Testing prediction for fips_code=4003
[DEBUG] Got prediction for fips_code=4003: length=4

[DEBUG] Testing prediction for fips_code=4001
[DEBUG] Prediction values:
[[ 0.00278093]
 [-0.01505762]
 [-0.02326078]
 [ 0.08019005]]

[DEBUG] Testing prediction for fips_code=4005
[DEBUG] Got prediction for fips_code=4001: length=4
[DEBUG] Prediction values:
[[-0.00711255]
 [-0.00882732]
 [-0.01959416]
 [ 0.00099299]]
[DEBUG] Got prediction for fips_code=4005: length=4
[DEBUG] Prediction values:
[[ 0.006242  ]
 [-0.01174934]
 [-0.00872862]
 [ 0.00469068]]

[DEBUG] Testing prediction for fips_code=4007
[DEBUG] Got prediction for fips_code=4007: length=4
[DEBUG] Prediction values:
[[ 0.00674986]
 [ 0.00083857]
 [-0.00849574]
 [ 0.00049601]]

[DEBUG] Testing prediction for fips_code=4009
[DEBUG] Got prediction for fips_code=4009: length=4
[DEBUG] Prediction values:
[[ 0.00165747]
 [-0.01825355]
 [-0.02511141]
 [ 0.07229721]]

[DEBUG] Testing prediction for fips_code=4019
[DEBUG] Got predic

Exception ignored in: <function ResourceTracker.__del__ at 0x10750aa20>
Traceback (most recent call last):
  File "/opt/homebrew/anaconda3/envs/erdos_spring_2025/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/homebrew/anaconda3/envs/erdos_spring_2025/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/opt/homebrew/anaconda3/envs/erdos_spring_2025/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes



[DEBUG] Testing prediction for fips_code=23001
[DEBUG] Got prediction for fips_code=23001: length=4
[DEBUG] Prediction values:
[[0.00443702]
 [0.01892184]
 [0.00898633]
 [0.12932625]]

[DEBUG] Testing prediction for fips_code=23003
[DEBUG] Got prediction for fips_code=23003: length=4
[DEBUG] Prediction values:
[[ 0.00091636]
 [ 0.02282647]
 [-0.00120429]
 [ 0.00256733]]


/opt/homebrew/anaconda3/envs/erdos_spring_2025/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


...


[DEBUG] Testing prediction for fips_code=45091
[DEBUG] Got prediction for fips_code=45091: length=4
[DEBUG] Prediction values:
[[0.01030662]
 [0.01691954]
 [0.0181024 ]
 [0.01789305]]

[DEBUG] Testing prediction for fips_code=4001
[DEBUG] Got prediction for fips_code=4001: length=4
[DEBUG] Prediction values:
[[0.00888751]
 [0.01163646]
 [0.01480907]
 [0.01145535]]

[DEBUG] Testing prediction for fips_code=4003
[DEBUG] Got prediction for fips_code=4003: length=4
[DEBUG] Prediction values:
[[ 0.00259885]
 [ 0.00211923]
 [-0.00030249]
 [ 0.00156354]]

[DEBUG] Testing prediction for fips_code=4005
[DEBUG] Got prediction for fips_code=4005: length=4
[DEBUG] Prediction values:
[[0.01203067]
 [0.01027678]
 [0.01200751]
 [0.00919083]]

[DEBUG] Testing prediction for fips_code=4007
[DEBUG] Got prediction for fips_code=4007: length=4
[DEBUG] Prediction values:
[[0.01598311]
 [0.01960652]
 [0.01326337]
 [0.01423602]]

[DEBUG] Testing prediction for fips_code=4009
[DEBUG] Got prediction for fips_

CV Folds: 100%|██████████| 5/5 [05:11<00:00, 62.31s/it]


[DEBUG] Testing prediction for fips_code=45091
[DEBUG] Got prediction for fips_code=45091: length=4
[DEBUG] Prediction values:
[[0.00934228]
 [0.01877396]
 [0.01179303]
 [0.01067747]]


(   fips_code  fold  predicted_max true_max            datetime
 0       4001     0       0.000993     None 2020-05-02 06:00:00
 1       4003     0       0.080190     None 2020-05-02 06:00:00
 2       4005     0       0.006242     None 2020-05-02 06:00:00
 3       4007     0       0.006750     None 2020-05-02 06:00:00
 4       4009     0       0.072297     None 2020-05-02 06:00:00,
 {'objective': 'reg:squarederror',
  'base_score': None,
  'booster': None,
  'callbacks': None,
  'colsample_bylevel': None,
  'colsample_bynode': None,
  'colsample_bytree': None,
  'device': None,
  'early_stopping_rounds': None,
  'enable_categorical': False,
  'eval_metric': None,
  'feature_types': None,
  'gamma': None,
  'grow_policy': None,
  'importance_type': None,
  'interaction_constraints': None,
  'learning_rate': None,
  'max_bin': None,
  'max_cat_threshold': None,
  'max_cat_to_onehot': None,
  'max_delta_step': None,
  'max_depth': None,
  'max_leaves': None,
  'min_child_weight': None,
  

Exception ignored in: <function ResourceTracker.__del__ at 0x107d16a20>
Traceback (most recent call last):
  File "/opt/homebrew/anaconda3/envs/erdos_spring_2025/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/homebrew/anaconda3/envs/erdos_spring_2025/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/opt/homebrew/anaconda3/envs/erdos_spring_2025/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x108816a20>
Traceback (most recent call last):
  File "/opt/homebrew/anaconda3/envs/erdos_spring_2025/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/homebrew/anaconda3/envs/erdos_spring_2025/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/opt/homebrew/anaconda3/envs/erdos_spring_2025/lib/python3.12/multiprocessing/resource_